# Residual stream diagnostics - INIT (Born) vs FIT

Geometry-selectable notebook for **PeriodicMaxwellNet** (UNet). Not tied to one wandb run.

**Demo comparison**

| View | Weights | Expected signature |
|---|---|---|
| **INIT** | fresh `PeriodicMaxwellNet` (zero `last_conv`) | envelope `a=0`, `E=E_i` (Born), flat `head` |
| **FIT** | `notebooks/latest.pt` | structured envelope; Helmholtz residual should drop |

Change `SAMPLE_IDS` / `THETAS_DEG` below to inspect other geometries.


## Physics primer (unchanged architecture)

- Network predicts envelope `a` with **total field** `E = E_i * (1 + a)`.
- **INIT is intentionally blank at the head:** `last_conv` is zero-initialized so training starts at the Born approximation (`a=0`, `E=E_i`). A flat `head` panel under INIT is correct.
- **k-injection (not redesigned):** `(kx, kz)` from incidence angle / wavelength
  1. build the plane-wave carrier `E_i = exp(i(kx x + kz z))`, and
  2. are broadcast to the bottleneck spatial grid and **concatenated as 2 channels** before the deepest UNet block (`UNet.forward`).
- Contrast input is scaled `(eps - 1) * contrast_scale`; true `eps` is used in the Helmholtz residual.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

NOTEBOOK_DIR = Path(".").resolve()
if NOTEBOOK_DIR.name != "notebooks":
    candidate = NOTEBOOK_DIR / "notebooks"
    if candidate.is_dir():
        NOTEBOOK_DIR = candidate.resolve()

REPO_ROOT = NOTEBOOK_DIR.parent
for path in (NOTEBOOK_DIR, REPO_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# --- geometry selectors (edit these) ---
SAMPLE_IDS = ["sample_0000"]
THETAS_DEG = (0.0, 45.0)

CKPT_PATH = NOTEBOOK_DIR / "latest.pt"
PML_THICKNESS = 10

import torch

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"notebook_dir={NOTEBOOK_DIR}")
print(f"device={DEVICE}")
print(f"ckpt={CKPT_PATH} exists={CKPT_PATH.is_file()}")
print(f"SAMPLE_IDS={SAMPLE_IDS}  THETAS_DEG={THETAS_DEG}")


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset

from ckpt_io import build_fit_net, build_init_net, list_sibling_snapshots
from maxwell_losses.helmholtz_checker import helmholtz_residual_loss_periodic_pml
from train_maxwellnet import _fem_residual, _plot_angle_group
from unet_activations import (
    STAGE_LABELS,
    UNetPathProbe,
    forward_with_activations,
    path_order,
    stage_title,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "image.cmap": "magma"})
torch.set_grad_enabled(False)


## 1. Load INIT and FIT weights


In [ ]:
if not CKPT_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {CKPT_PATH}. Place a train_maxwellnet checkpoint there "
        "(e.g. copy test_run1/model/latest.pt)."
    )

net_fit, fit_info = build_fit_net(CKPT_PATH, DEVICE)
mode = fit_info.mode
network_specs = dict(fit_info.network_specs)
net_init = build_init_net(mode, network_specs, DEVICE)

probe_init = UNetPathProbe(net_init)
probe_fit = UNetPathProbe(net_fit)

WEIGHT_LABEL_INIT = "weights=INIT (Born a=0, zero last_conv)"
WEIGHT_LABEL_FIT = (
    f"weights=FIT epoch={fit_info.epoch}  file={fit_info.path.name}"
    + (f"  wandb={fit_info.wandb_run_id}" if fit_info.wandb_run_id else "")
)

print(WEIGHT_LABEL_INIT)
print(WEIGHT_LABEL_FIT)
print("mode:", mode, "NetworkSpecs:", network_specs)
print(f"params={sum(p.numel() for p in net_fit.parameters()):,}")

siblings = list_sibling_snapshots(CKPT_PATH, mode=mode)
print(f"sibling epoch snaps next to ckpt: {len(siblings)}")
for p in siblings[:12]:
    print(" ", p.name)
if len(siblings) > 12:
    print("  ...")


## 2. Load selected geometries

Single parquet shard only (avoids the multi-shard HF train glob).


In [ ]:
def decode_row(row: dict) -> dict:
    """HF row -> ShapeDataset-style dict (257->256 crop)."""
    optical = torch.complex(
        torch.as_tensor(row["optical_constant_real"]),
        torch.as_tensor(row["optical_constant_imag"]),
    )[:-1, :-1, :]
    e_field = torch.complex(
        torch.as_tensor(row["e_field_real"]),
        torch.as_tensor(row["e_field_imag"]),
    )[:-1, :-1, :]
    return {
        "sample_id": row["sample_id"],
        "optical_constant": optical,
        "e_field": e_field,
        "wavelength_nm": float(row["wavelength_nm"]),
        "pol": row["pol"],
        "theta": float(row["theta"]),
        "delta_x_a": 1.0,
        "delta_z_a": 1.0,
    }


def batchify(row: dict) -> dict:
    """Add batch dim for PeriodicMaxwellNet.forward."""
    out = {}
    for key, value in row.items():
        if torch.is_tensor(value):
            out[key] = value.unsqueeze(0)
        elif key in ("sample_id", "pol"):
            out[key] = [value]
        else:
            out[key] = torch.tensor([value])
    return out


pol = "s" if mode == "te" else "p"
wanted_ids = set(SAMPLE_IDS)
wanted_th = set(float(t) for t in THETAS_DEG)

ds = load_dataset(
    "als-rixs/latent-image-training",
    data_files="training/train-00000-of-00001.parquet",
    split="train",
    revision="fixed-dimension",
)

rows: dict[tuple[str, float], dict] = {}
for i, (sid, p, th) in enumerate(zip(ds["sample_id"], ds["pol"], ds["theta"])):
    th = float(th)
    if sid in wanted_ids and p == pol and th in wanted_th:
        key = (sid, th)
        if key not in rows:
            rows[key] = decode_row(ds[i])
            print(f"row {i}: {sid} pol={p} theta={th:g}")

missing = [(sid, th) for sid in SAMPLE_IDS for th in THETAS_DEG if (sid, float(th)) not in rows]
if missing:
    raise LookupError(f"missing geometries {missing}")

batches = {key: batchify(row) for key, row in rows.items()}
GEOM_KEYS = [(sid, float(th)) for sid in SAMPLE_IDS for th in THETAS_DEG]
print("loaded", len(batches), "geometry batches")


## 3. Forward INIT and FIT on all selected geometries


In [ ]:
def run_geometry(
    key: tuple[str, float],
    model,
    act_probe: UNetPathProbe,
) -> dict:
    """Forward one (sample_id, theta) and attach Helmholtz residual diagnostics."""
    batch = batches[key]
    out = forward_with_activations(
        model,
        act_probe,
        batch["optical_constant"].to(torch.complex64).to(DEVICE),
        batch["theta"].to(torch.float32).to(DEVICE),
        batch["wavelength_nm"].to(torch.float32).to(DEVICE),
        batch["delta_x_a"].to(torch.float32).to(DEVICE),
        batch["delta_z_a"].to(torch.float32).to(DEVICE),
    )
    wavelength_a_real = float(batch["wavelength_nm"][0]) * 10.0
    dx = float(batch["delta_x_a"][0]) / wavelength_a_real
    dz = float(batch["delta_z_a"][0]) / wavelength_a_real
    pol_name = batch["pol"][0]
    residual = helmholtz_residual_loss_periodic_pml(
        out["envelope"].to(DEVICE),
        out["eps"].to(DEVICE),
        out["incident"].to(DEVICE),
        out["kz"].to(DEVICE) * wavelength_a_real,
        out["kx"].to(DEVICE) * wavelength_a_real,
        pol_name,
        1.0,
        dx,
        dz,
        PML_THICKNESS,
    ).detach().cpu()
    residual_gt = _fem_residual(
        batch["e_field"],
        out["eps"],
        out["incident"],
        out["kz"] * wavelength_a_real,
        out["kx"] * wavelength_a_real,
        mode,
        1.0,
        dx,
        dz,
        PML_THICKNESS,
    ).detach().cpu()
    return {
        **out,
        "residual": residual,
        "residual_gt": residual_gt,
        "sample_id": key[0],
        "theta": key[1],
        "mean_res2": float(residual.abs().pow(2).mean()),
        "mean_res2_gt": float(residual_gt.abs().pow(2).mean()),
        "mean_abs_envelope": float(out["envelope"].abs().mean()),
        "mean_abs_head": float(out["activations"]["head"].tensor.abs().mean()),
    }


results_init = {}
results_fit = {}
for key in GEOM_KEYS:
    results_init[key] = run_geometry(key, net_init, probe_init)
    results_fit[key] = run_geometry(key, net_fit, probe_fit)
    ri, rf = results_init[key], results_fit[key]
    print(
        f"{key[0]} theta={key[1]:g}:  "
        f"INIT |r|^2={ri['mean_res2']:.4e}  |a|={ri['mean_abs_envelope']:.4e}  |head|={ri['mean_abs_head']:.4e}  |  "
        f"FIT  |r|^2={rf['mean_res2']:.4e}  |a|={rf['mean_abs_envelope']:.4e}  |head|={rf['mean_abs_head']:.4e}  "
        f"FEM floor={rf['mean_res2_gt']:.4e}"
    )


## A. Helmholtz residuals - INIT vs FIT

Field / residual panels (wandb-style) for each geometry, INIT row then FIT row.
Summary table of mean `|residual|^2`.


In [ ]:
summary_rows = []
for key in GEOM_KEYS:
    sid, th = key
    for label, res, wlabel in (
        ("INIT", results_init[key], WEIGHT_LABEL_INIT),
        ("FIT", results_fit[key], WEIGHT_LABEL_FIT),
    ):
        summary_rows.append(
            {
                "view": label,
                "sample_id": sid,
                "theta": th,
                "mean_|r|^2": res["mean_res2"],
                "FEM_floor": res["mean_res2_gt"],
                "mean_|a|": res["mean_abs_envelope"],
                "mean_|head|": res["mean_abs_head"],
            }
        )
        fig = _plot_angle_group(
            [0],
            res["envelope"],
            res["residual"],
            res["residual_gt"],
            res["incident"],
            batches[key]["e_field"],
            [sid],
            mode,
        )
        fig.suptitle(f"{label}  {sid}  theta={th:g}\n{wlabel}", fontsize=11)
        plt.show()

print(f"{'view':5s}  {'sample':12s}  {'theta':>6s}  {'mean|r|^2':>12s}  {'FEM_floor':>12s}  {'mean|a|':>10s}  {'mean|head|':>12s}")
for row in summary_rows:
    print(
        f"{row['view']:5s}  {row['sample_id']:12s}  {row['theta']:6.1f}  "
        f"{row['mean_|r|^2']:12.4e}  {row['FEM_floor']:12.4e}  "
        f"{row['mean_|a|']:10.4e}  {row['mean_|head|']:12.4e}"
    )

# Compact bar comparison of mean |r|^2
fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
x = np.arange(len(GEOM_KEYS))
w = 0.35
ax.bar(x - w / 2, [results_init[k]["mean_res2"] for k in GEOM_KEYS], w, label="INIT")
ax.bar(x + w / 2, [results_fit[k]["mean_res2"] for k in GEOM_KEYS], w, label="FIT")
ax.plot(x, [results_fit[k]["mean_res2_gt"] for k in GEOM_KEYS], "k--", marker="o", label="FEM floor")
ax.set_xticks(x)
ax.set_xticklabels([f"{k[0]}\ntheta={k[1]:g}" for k in GEOM_KEYS], fontsize=8)
ax.set_yscale("log")
ax.set_ylabel("mean |residual|^2")
ax.set_title("Helmholtz residual: INIT vs FIT")
ax.legend(fontsize=8)
plt.show()


## B. UNet activation path - INIT vs FIT

Channel-mean `|activation|` along encoder → bottleneck → decoder → head.
INIT `head` should be ~0; FIT `head` should show spatial structure.


In [ ]:
def plot_activation_path(result: dict, weight_label: str, *, title_prefix: str) -> None:
    """Plot contrast + UNet stage energies for one forward."""
    contrast = result["contrast"][0]
    names = [n for n in path_order(result["activations"]) if not n.startswith("pool_")]
    panels = [
        ("contrast Re", contrast[0].numpy()),
        ("contrast Im", contrast[1].numpy()),
    ]
    for name in names:
        panels.append((name, result["activations"][name].energy[0].numpy()))

    ncols = 4
    nrows = int(math.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.1 * nrows), constrained_layout=True)
    axes = np.atleast_2d(axes)
    for i, (key, img) in enumerate(panels):
        ax = axes[i // ncols, i % ncols]
        im = ax.imshow(img, origin="lower", aspect="equal")
        ax.set_title(stage_title(key, img.shape), fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046)
    for j in range(len(panels), nrows * ncols):
        axes[j // ncols, j % ncols].axis("off")
    fig.suptitle(
        f"{title_prefix}  {result['sample_id']}  theta={result['theta']:g}\n{weight_label}",
        fontsize=11,
    )
    plt.show()


STAGE_KEYS = ["enc_0", "enc_2", "bottleneck", "dec_1", "dec_3", "head"]
for key in GEOM_KEYS:
    plot_activation_path(results_init[key], WEIGHT_LABEL_INIT, title_prefix="INIT")
    plot_activation_path(results_fit[key], WEIGHT_LABEL_FIT, title_prefix="FIT")

# Path magnitude comparison for the first geometry
key0 = GEOM_KEYS[0]
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for label, res in (("INIT", results_init[key0]), ("FIT", results_fit[key0])):
    ys = [float(res["activations"][k].tensor.abs().mean()) for k in STAGE_KEYS]
    ax.plot(range(len(STAGE_KEYS)), ys, marker="o", label=label)
ax.set_xticks(range(len(STAGE_KEYS)))
ax.set_xticklabels([STAGE_LABELS.get(k, k).split(":")[0] for k in STAGE_KEYS], rotation=25, ha="right")
ax.set_ylabel("mean |activation|")
ax.set_yscale("log")
ax.set_title(f"UNet path magnitudes - {key0[0]} theta={key0[1]:g}")
ax.legend()
plt.show()


## C. k-injection / bottleneck conditioning

`(kx, kz)` are concatenated at the bottleneck only (architecture unchanged).
Compare bottleneck energy INIT vs FIT and angle sensitivity under FIT.


In [ ]:
print("Incidence conditioning (from FIT forward):")
for key in GEOM_KEYS:
    cond = results_fit[key]["cond"][0].tolist()
    print(f"  {key[0]} theta={key[1]:g}: (kx, kz) = ({cond[0]:.6e}, {cond[1]:.6e}) Angstrom^-1")

# Bottleneck energy grids: INIT vs FIT for each geometry
n = len(GEOM_KEYS)
fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 6.2), constrained_layout=True)
if n == 1:
    axes = np.array([[axes[0]], [axes[1]]])
for col, key in enumerate(GEOM_KEYS):
    e_i = results_init[key]["activations"]["bottleneck"].energy[0].numpy()
    e_f = results_fit[key]["activations"]["bottleneck"].energy[0].numpy()
    for row, (img, lab) in enumerate(((e_i, "INIT"), (e_f, "FIT"))):
        ax = axes[row, col]
        im = ax.imshow(img, origin="lower", aspect="equal")
        ax.set_title(f"{lab}\n{key[0]} theta={key[1]:g}", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("Bottleneck energy (after kx,kz concat)\n" + WEIGHT_LABEL_FIT, fontsize=11)
plt.show()

# Angle sensitivity under FIT when at least two thetas share a sample_id
by_sample: dict[str, list[float]] = {}
for sid, th in GEOM_KEYS:
    by_sample.setdefault(sid, []).append(th)

for sid, thetas in by_sample.items():
    if len(thetas) < 2:
        continue
    th0, th1 = thetas[0], thetas[1]
    z0 = results_fit[(sid, th0)]["activations"]["bottleneck"].tensor[0]
    z1 = results_fit[(sid, th1)]["activations"]["bottleneck"].tensor[0]
    e0 = z0.abs().mean(0).numpy()
    e1 = z1.abs().mean(0).numpy()
    fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.2), constrained_layout=True)
    panels = [
        (e0, f"FIT theta={th0:g} energy"),
        (e1, f"FIT theta={th1:g} energy"),
        (np.abs(e1 - e0), "|energy diff|"),
        ((z1 - z0).abs().mean(0).numpy(), "mean_c |z1 - z0|"),
    ]
    for ax, (img, title) in zip(axes, panels):
        im = ax.imshow(img, origin="lower", aspect="equal")
        ax.set_title(title, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046)
    rel = float(torch.linalg.vector_norm(z1 - z0) / (torch.linalg.vector_norm(z0) + 1e-12))
    print(f"{sid} FIT bottleneck ||z(theta={th1})-z(theta={th0})|| / ||z0|| = {rel:.4e}")
    fig.suptitle(f"{sid} bottleneck angle sensitivity (FIT)\n{WEIGHT_LABEL_FIT}", fontsize=11)
    plt.show()


## Optional: channel peek on FIT bottleneck

Uses the first selected geometry under FIT weights.


In [ ]:
key = GEOM_KEYS[0]
name = "bottleneck"
chw = results_fit[key]["activations"][name].tensor[0]
nshow = min(4, chw.shape[0])
fig, axes = plt.subplots(1, nshow, figsize=(3 * nshow, 2.9), constrained_layout=True)
if nshow == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    im = ax.imshow(chw[i].numpy(), origin="lower", aspect="equal")
    ax.set_title(f"channel {i}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(
    f"FIT {stage_title(name, tuple(chw.shape))}  {key[0]} theta={key[1]:g}\n{WEIGHT_LABEL_FIT}",
    fontsize=11,
)
plt.show()
